In [ ]:
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

ROOT = "."
PRED_PATH = "predictions/TFT/TFT_dyn_5km_40_pred.parquet"
OBS_PATH  = os.path.join(ROOT, "data", "sample.csv")
CUTOFF    = pd.Timestamp("2024-01-01")

pred = pq.read_table(PRED_PATH).to_pandas()
pred = pred.rename(columns={"gws":"gws_forecast"})
gws_bb = pd.read_csv(OBS_PATH)
gws_bb["datum"] = pd.to_datetime(gws_bb["datum"])

lookup_ids = pd.DataFrame(gws_bb["id"].unique(), columns=["id"]).reset_index()

df = (pred.merge(lookup_ids, on="index")
          .merge(gws_bb[["id","datum","gws"]], on=["id","datum"], how="left")
          .rename(columns={"gws":"gws_true"}))

df["horizon"] = ((df["datum"] - df["startzeitpunkt"]) / pd.Timedelta(weeks=1)).astype(int) + 1

df["gws_forecast"] = df["gws_forecast"].astype(np.float32).round(2)
df["difference"] = (df["gws_forecast"] - df["gws_true"]).abs().round(2)

hist = gws_bb[gws_bb["datum"] < CUTOFF].copy()
hist["woy"] = hist["datum"].dt.isocalendar().week.astype(int)
clim = hist.groupby(["id","woy"], as_index=False)["gws"].mean().rename(columns={"gws":"clim_base"})

df["woy"] = df["datum"].dt.isocalendar().week.astype(int)
df = df.merge(clim, on=["id","woy"], how="left")

well_mean = hist.groupby("id")["gws"].mean().rename("well_mean")
df = df.merge(well_mean, on="id", how="left")
df["clim_base"] = df["clim_base"].fillna(df["well_mean"])
df.drop(columns=["well_mean"], inplace=True)

In [15]:
def mae(y, yhat): return np.mean(np.abs(yhat - y))
def rmse(y, yhat): return np.sqrt(np.mean((yhat - y)**2))
def nse(y, yhat):
    denom = np.sum((y - np.mean(y))**2)
    return np.nan if denom == 0 else 1 - np.sum((yhat - y)**2) / denom
def kge(y, yhat):
    r = np.corrcoef(y, yhat)[0,1]
    alpha = np.std(yhat, ddof=1) / np.std(y, ddof=1)
    beta  = np.mean(yhat) / np.mean(y)
    return 1 - np.sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2)
def rmbe(y, yhat):
    mbe = np.mean(yhat - y)
    return mbe / np.mean(y)
def nrmse(y, yhat, how="range"):
    r = rmse(y, yhat)
    if how == "range":   denom = np.max(y) - np.min(y)
    elif how == "mean":  denom = np.mean(y)
    elif how == "std":   denom = np.std(y, ddof=1)
    else: raise ValueError("how must be 'range','mean','std'")
    return np.nan if denom == 0 else r / denom


mask = np.isfinite(df["gws_true"]) & np.isfinite(df["gws_forecast"])
y, yhat = df.loc[mask, "gws_true"].to_numpy(), df.loc[mask, "gws_forecast"].to_numpy()
metrics = {
    "KGE":  kge(y, yhat),
    "MAE":  mae(y, yhat),
    "NSE":  nse(y, yhat),
    "RMSE": rmse(y, yhat),
    "NRMSE_range": nrmse(y, yhat, "range"),
    "RMBE": rmbe(y, yhat),
}
pd.Series(metrics)

KGE            0.999694
MAE            0.055127
NSE            0.999983
RMSE           0.080789
NRMSE_range    0.001750
RMBE           0.000194
dtype: float64

In [25]:
def skill(group):
    rmse_model = rmse(group["gws_true"], group["gws_forecast"])
    rmse_clim = rmse(group["gws_true"], group["clim_base"])
    return np.nan if rmse_clim == 0 else 1 - rmse_model / rmse_clim

test = df[df["datum"] >= CUTOFF].copy()
by_hor = test.groupby("horizon", dropna=False).apply(skill)
by_well = test.groupby("id", dropna=False).apply(skill)

print("RMSE by horizon:")
pd.Series(by_hor.to_dict())

RMSE by horizon:


1    0.829162
2    0.776781
3    0.731038
4    0.690759
dtype: float64

In [29]:
print("Median of RMSE by well:") 
float(by_well.median())

Median of RMSE by well:


0.7885108510281091

In [21]:
df.head(20)

,datum,gws_forecast,index,startzeitpunkt,id,gws_true,horizon,difference,woy,clim_base
0,2019-11-11,78.97,0,2019-11-11,LFU_25470023,78.91,1,0.06,46,79.19
1,2019-11-18,78.95,0,2019-11-11,LFU_25470023,78.91,2,0.04,47,79.18
2,2019-11-25,78.96,0,2019-11-11,LFU_25470023,78.90,3,0.06,48,79.19
3,2019-12-02,78.96,0,2019-11-11,LFU_25470023,78.89,4,0.07,49,79.18
4,2019-11-18,78.95,0,2019-11-18,LFU_25470023,78.91,1,0.04,47,79.18
5,2019-11-25,78.96,0,2019-11-18,LFU_25470023,78.90,2,0.06,48,79.19
6,2019-12-02,78.95,0,2019-11-18,LFU_25470023,78.89,3,0.06,49,79.18
7,2019-12-09,78.96,0,2019-11-18,LFU_25470023,78.87,4,0.09,50,79.17
8,2019-11-25,78.96,0,2019-11-25,LFU_25470023,78.90,1,0.06,48,79.19
9,2019-12-02,78.96,0,2019-11-25,LFU_25470023,78.89,2,0.07,49,79.18
